EGM722 Programming for GIS and Remote Sensing Assessment 2

Title: GIS Assessment of Geothermal Energy Potential and Environmental Constraints in Northern Ireland.

Aim: This project compares potentially suitable geothermal areas with environmentally protected areas in Northern Ireland.

1 Import Python modules

In [1]:
import sys

!{sys.executable} -m pip install geopandas folium

2 Import Python libraries

In [2]:
# File and folder handling
from pathlib import Path
import os
import zipfile
import urllib.request

# Data analysis
import pandas as pd

# Spatial data handling
import geopandas as gpd

# Mapping and plotting
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

# Interactive web mapping
import folium

# Spatial geometry operations
from shapely.geometry import box

# Suppress warnings for cleaner notebook output
import warnings
warnings.filterwarnings("ignore")

# Display all dataframe columns in notebook outputs
pd.set_option("display.max_columns", 100)

3 Set file paths

This section defines the folder structure used throughout the project.

Separate folders are used for:
- raw downloaded datasets,
- processed GIS outputs,
- and exported figures/results.

Using a consistent folder structure improves reproducibility and organisation of spatial analysis workflows. The folders are automatically created if they do not already exist.

The project uses pathlib.Path because it provides a platform-independent and reliable method for handling file paths in Python.

In [3]:
from pathlib import Path

# Project folders
PROJECT_DIR = Path.cwd().parent

DATA_RAW = PROJECT_DIR / "data_raw"
DATA_PROCESSED = PROJECT_DIR / "data_processed"
OUTPUTS = PROJECT_DIR / "outputs"

# Create folders if needed
for folder in [DATA_RAW, DATA_PROCESSED, OUTPUTS]:
    folder.mkdir(exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Raw data folder:", DATA_RAW)
print("Processed data folder:", DATA_PROCESSED)
print("Outputs folder:", OUTPUTS)

Project directory: C:\Users\User\EGM722_Assessment2_Diane_Gibson
Raw data folder: C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw
Processed data folder: C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_processed
Outputs folder: C:\Users\User\EGM722_Assessment2_Diane_Gibson\outputs


4 Set Coordinate Reference Systems (CRS)

In [4]:
# British National Grid is used by the BGS deep geothermal shapefile
BNG_CRS = "EPSG:27700"

# Irish National Grid is useful for NI environmental datasets
ING_CRS = "EPSG:29903"

# WGS84 is required for Folium web maps
WEB_CRS = "EPSG:4326"

# Use BNG as the analysis CRS because the BGS geothermal layer is supplied in BNG
TARGET_CRS = BNG_CRS

5 Define resusable functions

In [15]:
def download_and_extract(url, zip_path, extract_folder):
    """
    Download a zipped GIS dataset and extract it locally.

    Parameters
    ----------
    url : str
        Direct URL to the zipped dataset.
    zip_path : pathlib.Path
        Local path where the ZIP file will be saved.
    extract_folder : pathlib.Path
        Folder where the ZIP file will be extracted.

    Returns
    -------
    pathlib.Path
        Path to the extracted folder.
    """
    if not zip_path.exists():
        print(f"Downloading: {url}")
        urllib.request.urlretrieve(url, zip_path)
    else:
        print(f"Already downloaded: {zip_path.name}")

    extract_folder.mkdir(exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_folder)

    print(f"Extracted to: {extract_folder}")
    return extract_folder

def find_file(folder, extension):
    """
    Find the first file with a chosen extension inside a folder.

    Parameters
    ----------
    folder : pathlib.Path
        Folder to search.
    extension : str
        File extension pattern, for example '*.shp' or '*.gpkg'.

    Returns
    -------
    pathlib.Path
        Path to the first matching file.
    """
    files = list(folder.rglob(extension))

    if len(files) == 0:
        raise FileNotFoundError(f"No {extension} file found in {folder}")

    return files[0]

def load_vector(path, target_crs=TARGET_CRS, layer=None):
    """
    Load a vector GIS dataset and reproject it to a target CRS.

    Parameters
    ----------
    path : str or pathlib.Path
        Path to the GIS file, for example SHP, GeoJSON or GeoPackage.
    target_crs : str
        Coordinate reference system to reproject the dataset into.
    layer : str, optional
        Layer name to read when using a GeoPackage with multiple layers.

    Returns
    -------
    geopandas.GeoDataFrame
        Loaded and reprojected spatial dataset.
    """
    # Read the spatial file using GeoPandas.
    gdf = gpd.read_file(path, layer=layer)

    # Stop the workflow if the dataset has no CRS.
    if gdf.crs is None:
        raise ValueError(f"{path} has no CRS defined.")

    # Reproject to the chosen CRS so all datasets align spatially.
    return gdf.to_crs(target_crs)

def clean_geodataframe(gdf):
    """
    Clean a GeoDataFrame by removing missing geometries and repairing invalid ones.

    Parameters
    ----------
    gdf : geopandas.GeoDataFrame
        Input spatial dataset.

    Returns
    -------
    geopandas.GeoDataFrame
        Cleaned spatial dataset.
    """
    # Work on a copy so the original data is not modified accidentally.
    gdf = gdf.copy()

    # Remove rows without valid geometry.
    gdf = gdf[gdf.geometry.notna()]
    gdf = gdf[~gdf.geometry.is_empty]

    # Repair invalid polygon geometries using a zero-width buffer.
    gdf["geometry"] = gdf.geometry.buffer(0)

    return gdf

def classify_geothermal_play(row):
    """
    Assign an indicative geothermal suitability score using BGS geothermal play attributes.

    The scoring is based on general geothermal reasoning:
    sedimentary basins, hydrothermal systems, and fault/fracture-controlled settings
    are considered more favourable than poorly defined or low-permeability settings.

    Parameters
    ----------
    row : pandas.Series
        One row from the geothermal GeoDataFrame.

    Returns
    -------
    int
        Suitability score from 1 to 5, where 5 is highest.
    """
    text = " ".join([
        str(row.get("Geotherm_2", "")),
        str(row.get("GeologicHa", "")),
        str(row.get("GeologicCo", "")),
        str(row.get("BGSinforma", "")),
        str(row.get("Status", ""))
    ]).lower()

    score = 1

    # Sedimentary basins are often favourable for hydrothermal geothermal resources
    if "sedimentary" in text or "basin" in text:
        score += 2

    # Hydrothermal systems are directly relevant to geothermal production
    if "hydrothermal" in text:
        score += 1

    # Faults and fractures can increase permeability
    if "fault" in text or "fracture" in text:
        score += 1

    # Sandstone and limestone can be favourable aquifer lithologies
    if "sandstone" in text or "limestone" in text:
        score += 1

    # Granites may be relevant for deep geothermal heat but often need fracture permeability
    if "granite" in text:
        score += 1

    return min(score, 5)

def classify_score(score):
    """
    Convert a numeric score into a class.

    Parameters
    ----------
    score : float
        Suitability or sensitivity score.

    Returns
    -------
    str
        Class label.
    """
    if score >= 4:
        return "High"
    elif score >= 3:
        return "Moderate"
    elif score >= 2:
        return "Low"
    else:
        return "Very low"

6 Download and extract data

6.1 Store the web address for downloading Geothermal data

In [7]:
GEOTHERMAL_URL = "https://webservices.bgs.ac.uk/accessions/download/185538?fileName=Deep_Geothermal_Areas_v2.zip"

6.2 Download and extract Geothermal data

In [9]:
GEOTHERMAL_FOLDER = download_and_extract(
    GEOTHERMAL_URL,
    DATA_RAW / "Deep_Geothermal_Areas_v2.zip",
    DATA_RAW / "deep_geothermal_areas_v2"
)

# List extracted files
for file in GEOTHERMAL_FOLDER.rglob("*"):
    print(file)

Already downloaded: Deep_Geothermal_Areas_v2.zip
Extracted to: C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\deep_geothermal_areas_v2
C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\deep_geothermal_areas_v2\185538
C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\deep_geothermal_areas_v2\185538\Deep_Geothermal_Areas_v2
C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\deep_geothermal_areas_v2\185538\Deep_Geothermal_Areas_v2\Deep_Geothermal_Areas_V2.cpg
C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\deep_geothermal_areas_v2\185538\Deep_Geothermal_Areas_v2\Deep_Geothermal_Areas_V2.dbf
C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\deep_geothermal_areas_v2\185538\Deep_Geothermal_Areas_v2\Deep_Geothermal_Areas_V2.lyrx
C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\deep_geothermal_areas_v2\185538\Deep_Geothermal_Areas_v2\Deep_Geothermal_Areas_V2.prj
C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\deep_geothermal_areas_v2\185538\Deep_G

6.3 Locate the Geothermal shapefile from the zip file

In [11]:
# Use recursive global search and wildcard to search through the folder 
# to find all files ending with .shp, i.e., shapefiles
# returns the results as a list
shp_files = list(GEOTHERMAL_FOLDER.rglob("*.shp"))

print("Shapefiles found:")

# for loop prints the file path for the shapefiles found in the folder
for file in shp_files:
    print(file)

# Select the first shapefile in the list (expect there to be only one)
GEOTHERMAL_FILE = shp_files[0]

# Print the shapefile 
print("Using:", GEOTHERMAL_FILE)

Shapefiles found:
C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\deep_geothermal_areas_v2\185538\Deep_Geothermal_Areas_v2\Deep_Geothermal_Areas_V2.shp
Using: C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\deep_geothermal_areas_v2\185538\Deep_Geothermal_Areas_v2\Deep_Geothermal_Areas_V2.shp


6.4 Load Geothermal data into GeoPanda DataFrame

In [16]:
# Find shapefile in the extracted dataset folder using find_file function
GEOTHERMAL_FILE = find_file(GEOTHERMAL_FOLDER, "*.shp")

# Load the shapefile into GeoPandas using load_vector function
geothermal = load_vector(GEOTHERMAL_FILE)

# Clean the dataset using the clean_geodataframe function
geothermal = clean_geodataframe(geothermal)

# Print the number of features (rows) in the dataset
print("Geothermal records:", len(geothermal))

# Display the first 5 rows of the 'geothermal' GeoDataFrame
display(geothermal.head())

# Print all column names in the dataset
print(geothermal.columns)

Geothermal records: 79


,Partner1,Geothermal,Geotherm_1,Geographic,Geograph_1,CondConvTy,Geotherm_2,PlateTectS,GeologicHa,GeologicCo,References,BGSLexicon,BGSAgeDesc,BGSinforma,Status,geometry
0,BGS,UK_CD1_003,Eastern England Permo-Trias,UKE,UK,CD1,Intracratonic Basin,Intracratonic / Rift basin,Hydrothermal,Litho/biofacies controlled,Rollin et al. 1995 extend to south Newell 2023,SSG,Triassic,Permo-Trias sedimentary basins,"Concealed, basin extent","POLYGON ((541415.802 339691.89, 541328.121 338..."
1,BGS,UK_CD1_002,Cheshire Basin Permo-Trias,UKD,UK,CD1,Intracratonic Basin,Intracratonic / Rift basin,Hydrothermal,Litho/biofacies controlled,Rollin et al. 1995,SSG PUND,Permo-Trias,Permo-Trias sedimentary basins,"Exposed and concealed, basin extent","POLYGON ((392089.784 391844.571, 392961.611 38..."
2,BGS,UK_CD1_001,Worcester-Wessex basins Permo-Trias,UKK,UK,CD1,Intracratonic Basin\r\n,Intracratonic / Rift basin,Hydrothermal,Litho/biofacies controlled,Rollin et al. 1995 edge match edit 2024,SSG PUND,Permo-Trias,Permo-Trias sedimentary basins,"Exposed and concealed, basin extent","POLYGON ((325451.911 128139.642, 327294.261 12..."
3,BGS,UK_CD1_004,Northern Ireland Permo-Trias basins,UKN,UK,CD1,Intracratonic Basin,Intracratonic / Rift basin,Hydrothermal,Litho/biofacies controlled,Raine and Reay 2021,SSG PUND,Permo-Trias,Permo-Trias sedimentary basins,"Exposed and concealed, basin extent","POLYGON ((100755.454 513786.409, 99052.774 514..."
4,BGS,UK_CD1_005,Northern Ireland Permo-Trias basins,UKN,UK,CD1,Intracratonic Basin,Intracratonic / Rift basin,Hydrothermal,Litho/biofacies controlled,Raine and Reay 2021,SSG PUND,Permo-Trias,Permo-Trias sedimentary basins,"Exposed and concealed, basin extent","POLYGON ((96201.675 572543.706, 96279.994 5762..."


Index(['Partner1', 'Geothermal', 'Geotherm_1', 'Geographic', 'Geograph_1',
       'CondConvTy', 'Geotherm_2', 'PlateTectS', 'GeologicHa', 'GeologicCo',
       'References', 'BGSLexicon', 'BGSAgeDesc', 'BGSinforma', 'Status',
       'geometry'],
      dtype='str')


6.5 Score geothermal suitability

In [17]:
# Create a new column in the GeoDataFrame called 'geothermal score'
# Run the classify_geothermal_play() function on every row in the GeoDataFrame
geothermal["geothermal_score"] = geothermal.apply(
    classify_geothermal_play,
    axis=1 # apply function row by row instead of column by column
)

# Create another new column called 'geothermal class'
# Convert numeric scores into classed categories, e.g., low, moderate, and high
# Take each geothermal score and pass it into classify_score() function
geothermal["geothermal_class"] = geothermal["geothermal_score"].apply(
    classify_score
)

# Display selected columns in a formatted table to inspect outputs
display(
    geothermal[
        [
            "Geotherm_2",
            "GeologicHa",
            "GeologicCo",
            "BGSinforma",
            "geothermal_score",
            "geothermal_class"
        ]
    ].head()
)

,Geotherm_2,GeologicHa,GeologicCo,BGSinforma,geothermal_score,geothermal_class
0,Intracratonic Basin,Hydrothermal,Litho/biofacies controlled,Permo-Trias sedimentary basins,4,High
1,Intracratonic Basin,Hydrothermal,Litho/biofacies controlled,Permo-Trias sedimentary basins,4,High
2,Intracratonic Basin\r\n,Hydrothermal,Litho/biofacies controlled,Permo-Trias sedimentary basins,4,High
3,Intracratonic Basin,Hydrothermal,Litho/biofacies controlled,Permo-Trias sedimentary basins,4,High
4,Intracratonic Basin,Hydrothermal,Litho/biofacies controlled,Permo-Trias sedimentary basins,4,High


6.6 Store the web addresses for downloading protected area datasets

In [18]:
ASSI_URL = "https://www.daera-ni.gov.uk/sites/default/files/publications/daera/ASSI%20-%20Irish%20National%20Grid_5.zip"
SAC_URL = "https://www.daera-ni.gov.uk/sites/default/files/publications/daera/Special%20Areas%20of%20Conservation%20-%20Irish%20National%20Grid_1.zip"
SPA_URL = "https://www.daera-ni.gov.uk/sites/default/files/publications/daera/Special%20Protected%20Areas%20-%20Irish%20National%20Grid_1.zip"
RAMSAR_URL = "https://www.daera-ni.gov.uk/sites/default/files/publications/daera/Ramsar%20Sites%20-%20Irish%20National%20Grid_1.zip"
AONB_URL = "https://www.daera-ni.gov.uk/sites/default/files/publications/daera/AONB%20-%20Irish%20National%20Grid_0.zip"
NNR_URL = "https://www.daera-ni.gov.uk/sites/default/files/publications/daera/National%20Nature%20Reserves%20-%20Irish%20National%20Grid_1.zip"
WHS_URL = "https://www.daera-ni.gov.uk/sites/default/files/publications/daera/World_Heritage_Site%20-%20Irish%20National%20Grid.zip"

6.7 Download and extract protected area datasets

In [21]:
# Download and extract datasets
ASSI_FOLDER = download_and_extract(
    ASSI_URL, 
    DATA_RAW / "assi.zip", 
    DATA_RAW / "assi"
)

SAC_FOLDER = download_and_extract(
    SAC_URL, 
    DATA_RAW / "sac.zip", 
    DATA_RAW / "sac"
)

SPA_FOLDER = download_and_extract(
    SPA_URL, 
    DATA_RAW / "spa.zip", 
    DATA_RAW / "spa"
)

RAMSAR_FOLDER = download_and_extract(
    RAMSAR_URL, 
    DATA_RAW / "ramsar.zip", 
    DATA_RAW / "ramsar"
)

AONB_FOLDER = download_and_extract(
    AONB_URL, 
    DATA_RAW / "aonb.zip", 
    DATA_RAW / "aonb"
)

NNR_FOLDER = download_and_extract(
    NNR_URL, 
    DATA_RAW / "nnr.zip", 
    DATA_RAW / "nnr"
)

WHS_FOLDER = download_and_extract(
    WHS_URL, 
    DATA_RAW / "whs.zip", 
    DATA_RAW / "whs"
)

# Load shapefiles using the load_vector() function and the find_file() function
# find-file() function searches through the extracted folder and finds the shapefile
# load_vector() function loads the shapefile into GeoPandas and reprojects it
# Creates GeoDataFrames
assi = load_vector(find_file(ASSI_FOLDER, "*.shp"))
sac = load_vector(find_file(SAC_FOLDER, "*.shp"))
spa = load_vector(find_file(SPA_FOLDER, "*.shp"))
ramsar = load_vector(find_file(RAMSAR_FOLDER, "*.shp"))
aonb = load_vector(find_file(AONB_FOLDER, "*.shp"))
nnr = load_vector(find_file(NNR_FOLDER, "*.shp"))
whs = load_vector(find_file(WHS_FOLDER, "*.shp"))

# Clean and repair geometry using the clean_geodataframe() function
assi = clean_geodataframe(assi)
sac = clean_geodataframe(sac)
spa = clean_geodataframe(spa)
ramsar = clean_geodataframe(ramsar)
aonb = clean_geodataframe(aonb)
nnr = clean_geodataframe(nnr)
whs = clean_geodataframe(whs)

# Inspect GeoDataFrames by printing the number of polygons / features in the datasets
print("ASSI records:", len(assi))
print("SAC records:", len(sac))
print("SPA records:", len(spa))
print("Ramsar records:", len(ramsar))
print("AONB records:", len(aonb))
print("NNR records:", len(nnr))
print("WHS records:", len(whs))

Already downloaded: assi.zip
Extracted to: C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\assi
Already downloaded: sac.zip
Extracted to: C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\sac
Already downloaded: spa.zip
Extracted to: C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\spa
Already downloaded: ramsar.zip
Extracted to: C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\ramsar
Already downloaded: aonb.zip
Extracted to: C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\aonb
Already downloaded: nnr.zip
Extracted to: C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\nnr
Already downloaded: whs.zip
Extracted to: C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\whs
ASSI records: 394
SAC records: 58
SPA records: 16
Ramsar records: 20
AONB records: 8
NNR records: 50
WHS records: 1


6.8 Create environmental sensitivity scores

In [22]:
# Scores assigned based on assumed environmental planning sensitivity
# Higher scores indicate stronger environmental constraint
# SAC, SPA, Ramsar, NNR and WHS are assigned the maximum score because they represent internationally or European designated protected areas
# ASSI is scored slightly lower due to their national ecological and geological importance
# AONB is scored lower again because it is mainly a landscape designation

assi["constraint_type"] = "ASSI"
assi["environment_score"] = 4

sac["constraint_type"] = "SAC"
sac["environment_score"] = 5

spa["constraint_type"] = "SPA"
spa["environment_score"] = 5

ramsar["constraint_type"] = "Ramsar"
ramsar["environment_score"] = 5

aonb["constraint_type"] = "AONB"
aonb["environment_score"] = 3

nnr["constraint_type"] = "National Nature Reserve"
nnr["environment_score"] = 5

whs["constraint_type"] = "World Heritage Site"
whs["environment_score"] = 5

# Merge all protected-area layers into one combined constraints layer
protected_areas = pd.concat(
    [assi, sac, spa, ramsar, aonb, nnr, whs],
    ignore_index=True
)

# Convert the merged DataFrame back into a GeoDataFrame
protected_areas = gpd.GeoDataFrame(
    protected_areas,
    geometry="geometry",
    crs=TARGET_CRS
)

# Clean and repair geometries before spatial overlay
protected_areas = clean_geodataframe(protected_areas)

# Display the number of rows / features in the GeoDataFrame to check merge executed correctly
print("Combined protected-area records:", len(protected_areas))

# Display the first 5 rows of the GeoDataFrame in a formatted table  to inspect the columns / 
# attributes and the environmental sensitivity score assignment
display(protected_areas.head())

Combined protected-area records: 547


,OBJECTID,REFERENCE,NAME,COUNTY,MAP_SCALE,SPECIESPT1,SPECIESPT2,HABITAT,EARTH_SCI,PARTIES,DECLAREDAY,DECLARE_HA,CONFIRMDAY,CONFIRM_HA,GIS_AREA,GIS_LENGTH,Hyperlink,Shape_STAr,Shape_STLe,geometry,constraint_type,environment_score,OBJECTID_1,DEC_AREA,CALC_ARE,PERIMETER,GRID_REF,SUBMITTED,STATUS,CLASSIFIED,Latitude,Longitude,Status,Latitu,Longitu,Id,LEGISLATIN,DIST_COUNC,CALC_AREA,ID_REF,D_DEC_DATE,Area_Hecta,COMMENTS,TYPE,DESIGNATIO,HYPERLINK,Name_WHS,ISO3,LON,LAT,AREANAME,SITE_CODE,CONV_CODE,AREA_HA,CONV_FULL,CRITERIA,EST_DATE,Shape_ST_1,Shape_ST_2,MOD_DATE,MOD_CODE
0,4562.0,ASSI043,Rathlin Island - Ballygill North,Antrim,1:10000,Breeding bird assemblage,No ASSI feature present in this field category,Dry heath,No ASSI feature present in this field category,7.0,1991-09-16,78.00,1992-03-26,78.00,75.936678,4662.538215,https://www.daera-ni.gov.uk/publications/bally...,7.593668e+05,4662.538215,"POLYGON ((131508.196 609874.74, 131504.633 609...",ASSI,4,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaT,NaN
1,4563.0,ASSI044,Rathlin Island - Kinramer South,Antrim,1:10000,Pyramidal Bugle,No ASSI feature present in this field category,No ASSI feature present in this field category,No ASSI feature present in this field category,8.0,1991-09-16,25.00,1992-03-26,25.00,23.912271,2017.789353,https://www.daera-ni.gov.uk/publications/kinra...,2.391227e+05,2017.789353,"POLYGON ((129374.578 608267.756, 129370.917 60...",ASSI,4,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaT,NaN
2,4564.0,ASSI033,Rathlin Island - Coast,Antrim,1:10000,"Common Gull, Fulmar, Guillemot, Herring Gull, ...",No ASSI feature present in this field category,"Subtidal, Coastal vegetated shingle, Maritime ...","Tertiary igneous, Tertiary igneous",29.0,1991-09-16,257.00,1992-03-26,257.00,236.063540,77916.507670,https://www.daera-ni.gov.uk/publications/rathl...,2.360635e+06,77916.507670,"MULTIPOLYGON (((135544.721 608352.94, 135546.9...",ASSI,4,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaT,NaN
3,4565.0,ASSI050,Sheep Island,Antrim,1:10000,Great Cormorant,No ASSI feature present in this field category,No ASSI feature present in this field category,No ASSI feature present in this field category,2.0,1992-02-27,3.50,1992-07-08,3.50,3.495250,1282.752820,https://www.daera-ni.gov.uk/publications/sheep...,3.495250e+04,1282.752820,"POLYGON ((123429.816 603545.112, 123431.673 60...",ASSI,4,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaT,NaN
4,4566.0,ASSI116,Carrickarade,Antrim,1:10000,"Invertebrate assemblage, Breeding bird assemblage",No ASSI feature present in this field category,Maritime cliff and slopes,"Cretaceous stratigraphy, Tufa cascade, Tertiar...",4.0,1996-10-30,17.88,1997-03-13,17.88,17.799719,6939.272122,https://www.daera-ni.gov.uk/publications/carri...,1.779972e+05,6939.272122,"MULTIPOLYGON (((124946.553 602061.936, 124807....",ASSI,4,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaT,NaN


7 Analysis - Overlay geothermal areas with protected areas

In [23]:
# Identify where geothermal areas overlap with environmentally protected areas
# Creates a new GeoDataFrame called 'conflict_areas' which contains only the areas where both datasets overlap
conflict_areas = gpd.overlay(
    geothermal,
    protected_areas,
    how="intersection" # intersection keeps areas where the polygons overlap
)

# Calculate area of 'conflict_areas' in square kilometres
# Creates a new column 'area_km2' to store polygon area
conflict_areas["area_km2"] = conflict_areas.geometry.area / 1_000_000

# Print the number of overapping polygons created in 'conflict_areas' GeoDataFrame
# Check that the overlay worked, the intersection exists, and that the data loaded properly
print("Overlap records:", len(conflict_areas))

# Display the first 5 rows of the overlap datset in an output table
# Inspect the overlap attributes, scores, geometry, and calculated area
display(conflict_areas.head())

Overlap records: 294


,Partner1,Geothermal,Geotherm_1,Geographic,Geograph_1,CondConvTy,Geotherm_2,PlateTectS,GeologicHa,GeologicCo,References,BGSLexicon,BGSAgeDesc,BGSinforma,Status_1,geothermal_score,geothermal_class,OBJECTID,REFERENCE,NAME,COUNTY,MAP_SCALE,SPECIESPT1,SPECIESPT2,HABITAT,EARTH_SCI,PARTIES,DECLAREDAY,DECLARE_HA,CONFIRMDAY,CONFIRM_HA,GIS_AREA,GIS_LENGTH,Hyperlink,Shape_STAr,Shape_STLe,constraint_type,environment_score,OBJECTID_1,DEC_AREA,CALC_ARE,PERIMETER,GRID_REF,SUBMITTED,STATUS,CLASSIFIED,Latitude,Longitude,Status_2,Latitu,Longitu,Id,LEGISLATIN,DIST_COUNC,CALC_AREA,ID_REF,D_DEC_DATE,Area_Hecta,COMMENTS,TYPE,DESIGNATIO,HYPERLINK,Name_WHS,ISO3,LON,LAT,AREANAME,SITE_CODE,CONV_CODE,AREA_HA,CONV_FULL,CRITERIA,EST_DATE,Shape_ST_1,Shape_ST_2,MOD_DATE,MOD_CODE,geometry,area_km2
0,BGS,UK_CD1_004,Northern Ireland Permo-Trias basins,UKN,UK,CD1,Intracratonic Basin,Intracratonic / Rift basin,Hydrothermal,Litho/biofacies controlled,Raine and Reay 2021,SSG PUND,Permo-Trias,Permo-Trias sedimentary basins,"Exposed and concealed, basin extent",4,High,4583.0,ASSI143,Black Burn,Antrim,1:10000,No ASSI feature present in this field category,No ASSI feature present in this field category,No ASSI feature present in this field category,Cave,3.0,1996-12-11,20.3,1997-05-29,20.3,20.296762,3337.541031,https://www.daera-ni.gov.uk/publications/black...,2.029676e+05,3337.541031,ASSI,4,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaT,NaN,"POLYGON ((145271.889 577233.929, 145266.619 57...",0.164470
1,BGS,UK_CD1_004,Northern Ireland Permo-Trias basins,UKN,UK,CD1,Intracratonic Basin,Intracratonic / Rift basin,Hydrothermal,Litho/biofacies controlled,Raine and Reay 2021,SSG PUND,Permo-Trias,Permo-Trias sedimentary basins,"Exposed and concealed, basin extent",4,High,4587.0,ASSI002,Gortnagory,Antrim,1:10000,Irish Lady’s-tresses,No ASSI feature present in this field category,No ASSI feature present in this field category,No ASSI feature present in this field category,13.0,1986-12-17,4.2,1987-05-27,4.2,4.210800,823.059697,https://www.daera-ni.gov.uk/publications/gortn...,4.210800e+04,823.059697,ASSI,4,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaT,NaN,"POLYGON ((143951.022 575546.012, 144111.813 57...",0.042125
2,BGS,UK_CD1_004,Northern Ireland Permo-Trias basins,UKN,UK,CD1,Intracratonic Basin,Intracratonic / Rift basin,Hydrothermal,Litho/biofacies controlled,Raine and Reay 2021,SSG PUND,Permo-Trias,Permo-Trias sedimentary basins,"Exposed and concealed, basin extent",4,High,4592.0,ASSI067,Garron Plateau,Antrim,1:10000,"Higher plant assemblage, Invertebrate assembla...",No ASSI feature present in this field category,"Blanket bog, Fens, Dystrophic lakes, Oligotrop...",Tertiary igneous,101.0,1994-05-31,4650.0,1994-11-21,4650.0,4650.069419,52990.412821,https://www.daera-ni.gov.uk/publications/garro...,4.650069e+07,52990.412821,ASSI,4,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaT,NaN,"MULTIPOLYGON (((134425.503 571031.736, 135913....",38.710870
3,BGS,UK_CD1_004,Northern Ireland Permo-Trias basins,UKN,UK,CD1,Intracratonic Basin,Intracratonic / Rift basin,Hydrothermal,Litho/biofacies controlled,Raine and Reay 2021,SSG PUND,Permo-Trias,Permo-Trias sedimentary basins,"Exposed and concealed, basin extent",4,High,4593.0,ASSI026,Cleggan Valley,Antrim,1:10000,No ASSI feature present in this field category,No ASSI feature present in this field category,Mixed ashwoods,No ASSI feature present in this field category,4.0,1987-04-22,35.0,1987-11-12,35.0,36.775350,5051.077175,https://www.daera-ni.gov.uk/publications/clegg...,3.677535e+05,5051.077175,ASSI,4,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

8 Classify planning conflict

In [24]:
# Define a classification function 
# Function examines one row at a ttime, compares scores, and returns a text interpretation
def classify_conflict(row):
    """
    Classify overlap between geothermal potential and environmental sensitivity

    Parameters
    ----------
    row : pandas.Series
        Row containing geothermal and environmental scores

    Returns
    -------
    str
        Conflict interpretation category
    """
    geothermal_score = row["geothermal_score"] # read geothermal score from current row
    environment_score = row["environment_score"] # read environmental sensitivity score from the row

    # Decision rules
    # Rule 1
    if geothermal_score >= 4 and environment_score >= 4:
        return "High geothermal potential / high environmental constraint" 
    # if geothermal potential is high and environmental sensitivity is high then the statement 'high geothermal potential / high environmental constraint' will be returned
    # therefore, there is a potential planning conflict and constraint, and any geothermal project would require detailed environmental impact assessment
    # Rule 2
    elif geothermal_score >= 4 and environment_score < 4:
        return "High geothermal potential / lower environmental constraint"
    # potentially favourable devleopment areas with lower conflict zones
    # Rule 3
    elif geothermal_score >= 3 and environment_score >= 4:
        return "Moderate geothermal potential / high environmental constraint"
    # moderate geothermal opportunity but environmentally sensitive
    # therefore, geothermal project opportunities may be constrained, suitability is uncertain, and cautious planning is required
    # else statement
    else:
        return "Lower priority or lower conflict"
    # applies if none of the earlier conditions are met
    # indicates an area of lower geothermal suitability and lower environmental importance

# classify_conflict(row) function applied to each row
# Creates a new column called 'conflict_class'
conflict_areas["conflict_class"] = conflict_areas.apply(
    classify_conflict,
    axis=1 # runs the function on every row
)

# Display the first 5 rows of the selected columns in a formatted table to allow for inspection of results
display(
    conflict_areas[
        ["geothermal_class", "constraint_type", "environment_score",
         "conflict_class", "area_km2"]
    ].head()
)

,geothermal_class,constraint_type,environment_score,conflict_class,area_km2
0,High,ASSI,4,High geothermal potential / high environmental...,0.164470
1,High,ASSI,4,High geothermal potential / high environmental...,0.042125
2,High,ASSI,4,High geothermal potential / high environmental...,38.710870
3,High,ASSI,4,High geothermal potential / high environmental...,0.367940
4,High,ASSI,4,High geothermal potential / high environmental...,0.084315


9 Results table

In [25]:
# Create a DataFrame called 'summary_table' to contain grouped statistics and summarised GIS results
summary_table = (
    conflict_areas # select input data
    .groupby(["conflict_class", "constraint_type"]) # group data into categories
    .agg( # aggregate summary statistics for each group
        feature_count=("conflict_class", "count"), # count the number of polygons in each category
        total_area_km2=("area_km2", "sum"), # sum overlapping polygon areas
        mean_geothermal_score=("geothermal_score", "mean"), # calculate average geothermal suitability
        mean_environment_score=("environment_score", "mean") # calculate average environmental sensitivity
    )
    .reset_index() # convert grouped categories back into normal columns
)

# Display the summary table to inspect results and to interpret findings
display(summary_table)

# Export the table to a CSV file 
summary_table.to_csv(
    OUTPUTS / "geothermal_environmental_conflict_summary.csv",
    index=False # to avoid the creation of row-number column, which is not required
)

,conflict_class,constraint_type,feature_count,total_area_km2,mean_geothermal_score,mean_environment_score
0,High geothermal potential / high environmental...,ASSI,173,974.973229,4.624277,4.0
1,High geothermal potential / high environmental...,National Nature Reserve,27,16.003003,4.592593,5.0
2,High geothermal potential / high environmental...,Ramsar,12,1025.792239,4.583333,5.0
3,High geothermal potential / high environmental...,SAC,29,155.417912,4.586207,5.0
4,High geothermal potential / high environmental...,SPA,10,1018.237718,4.600000,5.0
5,High geothermal potential / high environmental...,World Heritage Site,2,1.019740,4.500000,5.0
6,High geothermal potential / lower environmenta...,AONB,8,758.648836,4.500000,3.0
7,Lower priority or lower conflict,AONB,4,330.734882,3.000000,3.0
8,Moderate geothermal potential / high environme...,ASSI,26,113.036467,3.000000,4.0
9,Moderate geothermal potential / high environme...,SAC,3,71.302639,3.000000,5.0


10 Outputs

10.1 Static map

10.2 Interactive map

10.3 Export GIS outputs

11 Conclusions and limitations